In [1]:
%pip install reportlab matplotlib -q


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import sys

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import TwoSlopeNorm
import warnings

warnings.filterwarnings("ignore")


def project_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "dataset" / "stock_market_19_24.csv").exists():
            return cand
    raise FileNotFoundError(
        "Không tìm thấy dataset/stock_market_19_24.csv. "
        "Mở Jupyter từ thư mục dự án gat_lstm_stock_prediction hoặc kiểm tra thư mục dataset."
    )


ROOT = project_root()
DATA_CSV = ROOT / "dataset" / "stock_market_19_24.csv"
FIG_DIR = ROOT / "dataset" / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load & clean data ─────────────────────────────────────────────────────────
df = pd.read_csv(DATA_CSV)
df["Date"] = pd.to_datetime(df["History Price / Date"])
df = df.sort_values("Date").reset_index(drop=True)
stock_cols = [c for c in df.columns if c not in ["History Price / Date", "Date"]]


def clean_price(x):
    if isinstance(x, str):
        x = x.replace(",", "").strip()
        if x in ["", "-", "nan"]:
            return np.nan
        return float(x)
    return x


prices = df[stock_cols].map(clean_price).ffill().bfill()
prices.index = pd.DatetimeIndex(df["Date"])

daily_ret = prices.pct_change().dropna()
weekly_p = prices.resample("W").last()
weekly_r = weekly_p.pct_change().dropna()
vol_ann = daily_ret.std() * np.sqrt(252) * 100   
norm = (prices / prices.iloc[0]) * 100          #Rebase to 100, chia mỗi cột cho 1st value của chính nó
corr = weekly_r.corr()

# Month-end (pandas 2.2+ prefers ME)
try:
    monthly = norm.resample("ME").last()    # lấy giá trị cuối mỗi tháng từ norm (đã Rebase)
except Exception:
    monthly = norm.resample("M").last()

# ── Style ─────────────────────────────────────────────────────────────────────
DARK = "#1a1a2e"
ACCENT1 = "#1D9E75"
ACCENT2 = "#378ADD"
ACCENT3 = "#BA7517"
ACCENT4 = "#D85A30"
GRAY = "#888780"
LIGHT = "#f5f5f0"
WHITE = "#ffffff"

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "figure.facecolor": WHITE,
        "axes.facecolor": WHITE,
        "axes.edgecolor": "#e0ddd8",
        "axes.grid": True,
        "grid.color": "#eeece8",
        "grid.linewidth": 0.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "text.color": "#2c2c2a",
        "axes.labelcolor": "#5f5e5a",
        "xtick.color": "#888780",
        "ytick.color": "#888780",
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlepad": 10,
    }
)

# FIGURE 1
fig1, ax = plt.subplots(figsize=(12, 5))
for s in stock_cols:
    if s not in ["HDC", "DIG", "PDR", "CCL", "VHM", "NVL", "HPX"]:
        ax.plot(monthly.index, monthly[s], color="#d0cec8", lw=0.8, alpha=0.6)

highlights = {
    "HDC": (ACCENT1, 2.5, "-"),
    "DIG": (ACCENT2, 2.2, "-"),
    "PDR": (ACCENT3, 1.8, "--"),
    "CCL": (ACCENT4, 1.8, "--"),
    "NVL": ("#7F77DD", 1.8, ":"),
    "HPX": ("#D4537E", 1.8, ":"),
    "VHM": (GRAY, 1.8, "-"),
}
for s, (col, lw, ls) in highlights.items():
    ax.plot(monthly.index, monthly[s], color=col, lw=lw, ls=ls, label=s, zorder=3)

for s, (col, lw, ls) in highlights.items():
    v = monthly[s].iloc[-1]
    ax.annotate(
        f"{s} ({v/100:.1f}x)",
        xy=(monthly.index[-1], v),
        xytext=(5, 0),
        textcoords="offset points",
        fontsize=7.5,
        color=col,
        va="center",
        fontweight="bold",
    )

regimes = [
    ("2020-02-01", "2020-04-15", "#EF9F27", "COVID-19\nSelloff"),
    ("2021-01-01", "2021-12-31", "#1D9E75", "Bubble\n2021"),
    ("2022-04-01", "2022-11-30", "#E24B4A", "Credit\nCrunch"),
]
for (s, e, c, lbl) in regimes:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=0.08, zorder=0)
    mid = pd.Timestamp(s) + (pd.Timestamp(e) - pd.Timestamp(s)) / 2
    ymax = monthly.max().max()
    ax.text(
        mid,
        ymax * 0.95,
        lbl,
        ha="center",
        va="top",
        fontsize=7,
        color=c,
        fontweight="bold",
        alpha=0.85,
    )

ax.axhline(100, color="#b0ada8", lw=0.8, ls="--", alpha=0.7)
ax.set_xlim(monthly.index[0], monthly.index[-1])
ax.set_ylabel("Rebased Price Index (Base = 100 at 28/06/2019)")
ax.set_title("Diễn biến giá 26 mã HOSE — Chuẩn hóa theo ngày đầu (06/2019 – 12/2024)")
ax.legend(loc="upper left", fontsize=7.5, framealpha=0.9, ncol=4, columnspacing=1)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x)}"))
plt.tight_layout()
fig1.savefig(FIG_DIR / "fig1_price_norm.png", dpi=180, bbox_inches="tight")
plt.close()
print("Fig1 saved ->", FIG_DIR / "fig1_price_norm.png")

# FIGURE 2
years = [2020, 2021, 2022, 2023, 2024]
ret_mat = np.zeros((len(stock_cols), len(years)))
for j, yr in enumerate(years):
    mask = prices.index.year == yr
    sub = prices[mask]
    if len(sub) > 5:
        r = (sub.iloc[-1] / sub.iloc[0] - 1) * 100
        for i, s in enumerate(stock_cols):
            ret_mat[i, j] = r[s]

fig2, ax = plt.subplots(figsize=(10, 8))
norm_cm = TwoSlopeNorm(vmin=-90, vcenter=0, vmax=330)
im = ax.imshow(ret_mat, cmap="RdYlGn", norm=norm_cm, aspect="auto")
ax.set_xticks(range(len(years)))
ax.set_xticklabels([str(y) for y in years], fontsize=10, fontweight="bold")
ax.set_yticks(range(len(stock_cols)))
ax.set_yticklabels(stock_cols, fontsize=8.5)
ax.set_title("Annual Return (%) — 26 mã × 5 năm\n(Xanh = sinh lời, Đỏ = thua lỗ)")
ax.grid(False)
for i in range(len(stock_cols)):
    for j in range(len(years)):
        v = ret_mat[i, j]
        txt_col = "white" if abs(v) > 60 else "#2c2c2a"
        ax.text(
            j,
            i,
            f"{v:+.0f}%",
            ha="center",
            va="center",
            fontsize=7.5,
            color=txt_col,
            fontweight="bold",
        )
cb = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cb.set_label("Return (%)", fontsize=9)
cb.ax.tick_params(labelsize=8)
for i, s in enumerate(stock_cols):
    avg = ret_mat[i].mean()
    color = ACCENT1 if avg > 0 else ACCENT4
    ax.text(
        len(years) + 0.15,
        i,
        f"avg: {avg:+.0f}%",
        va="center",
        fontsize=7,
        color=color,
        fontweight="bold",
    )
plt.tight_layout()
fig2.savefig(FIG_DIR / "fig2_annual_heatmap.png", dpi=180, bbox_inches="tight")
plt.close()
print("Fig2 saved")

# FIGURE 3
fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sorted_vol = vol_ann.sort_values(ascending=True)
colors_vol = [
    "#E24B4A" if v > 45 else "#BA7517" if v > 35 else "#378ADD" for v in sorted_vol
]
ax1.barh(range(len(sorted_vol)), sorted_vol.values, color=colors_vol, height=0.65, alpha=0.85)
ax1.set_yticks(range(len(sorted_vol)))
ax1.set_yticklabels(sorted_vol.index, fontsize=8.5)
ax1.set_xlabel("Volatility (% năm, annualized từ daily return)")
ax1.set_title("Xếp hạng Volatility — 26 mã\n(Đỏ >45%, Cam 35–45%, Xanh <35%)")
ax1.axvline(35, color="#BA7517", lw=1, ls="--", alpha=0.7)
ax1.axvline(45, color="#E24B4A", lw=1, ls="--", alpha=0.7)
for i, (s, v) in enumerate(sorted_vol.items()):
    ax1.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=7.5, color="#444441")
p1 = mpatches.Patch(color="#E24B4A", label=">45%: Rủi ro cao")
p2 = mpatches.Patch(color="#BA7517", label="35–45%: Trung bình")
p3 = mpatches.Patch(color="#378ADD", label="<35%: Ổn định")
ax1.legend(handles=[p1, p2, p3], fontsize=7.5, loc="lower right")
corr_vals = corr.values[np.triu_indices_from(corr.values, k=1)]
ax2.hist(corr_vals, bins=35, color=ACCENT2, alpha=0.7, edgecolor="white", linewidth=0.5)
ax2.axvline(corr_vals.mean(), color=ACCENT4, lw=2, ls="--", label=f"Mean = {corr_vals.mean():.3f}")
ax2.axvline(0.7, color="#E24B4A", lw=1.5, ls=":", label="Ngưỡng 0.7 (graph edge)")
ax2.set_xlabel("Pearson Correlation (weekly returns)")
ax2.set_ylabel("Số cặp mã")
ax2.set_title(
    f"Phân phối Correlation giữa 325 cặp mã\n(Mean={corr_vals.mean():.3f}, Std={corr_vals.std():.3f})"
)
ax2.legend(fontsize=8)
ax2.text(
    0.42,
    ax2.get_ylim()[1] * 0.85,
    f"{(corr_vals > 0).sum()} cặp\ncorr > 0",
    ha="center",
    fontsize=8,
    color=ACCENT1,
    fontweight="bold",
)
ax2.text(
    -0.02,
    ax2.get_ylim()[1] * 0.85,
    f"{(corr_vals < 0).sum()} cặp\ncorr < 0",
    ha="center",
    fontsize=8,
    color=ACCENT4,
    fontweight="bold",
)
ax2.text(
    0.75,
    ax2.get_ylim()[1] * 0.7,
    f"{(corr_vals >= 0.7).sum()} cặp\n≥ 0.7",
    ha="center",
    fontsize=8,
    color="#E24B4A",
    fontweight="bold",
)
plt.tight_layout()
fig3.savefig(FIG_DIR / "fig3_vol_corr.png", dpi=180, bbox_inches="tight")
plt.close()
print("Fig3 saved")

# FIGURE 4
fig4, axes = plt.subplots(1, 3, figsize=(14, 4.5))
all_wr = weekly_r.values.flatten()
all_wr = all_wr[~np.isnan(all_wr)]
ax = axes[0]
n, bins_e, patches = ax.hist(
    all_wr * 100, bins=60, color=ACCENT2, alpha=0.75, edgecolor="white", linewidth=0.3
)
for patch, left in zip(patches, bins_e[:-1]):
    patch.set_facecolor(ACCENT4 if left < 0 else ACCENT1)
    patch.set_alpha(0.75)
ax.axvline(0, color="#444441", lw=1.5, ls="--", alpha=0.7)
ax.axvline(
    np.mean(all_wr) * 100,
    color=ACCENT3,
    lw=2,
    ls="-",
    label=f"Mean={np.mean(all_wr)*100:.2f}%",
)
ax.set_xlabel("Weekly Return (%)")
ax.set_ylabel("Tần suất (observations)")
ax.set_title(f"Phân phối Weekly Return\n(N={len(all_wr):,} obs — 26 mã × 274 tuần)")
ax.legend(fontsize=8)
stats_txt = (
    f"Mean:  {np.mean(all_wr)*100:+.2f}%\n"
    f"Std:     {np.std(all_wr)*100:.2f}%\n"
    f"Min:    {np.min(all_wr)*100:.1f}%\n"
    f"Max:  {np.max(all_wr)*100:.1f}%\n"
    f"Skew:  {pd.Series(all_wr).skew():.2f}\n"
    f"Kurt:   {pd.Series(all_wr).kurt():.2f}"
)
ax.text(
    0.97,
    0.97,
    stats_txt,
    transform=ax.transAxes,
    fontsize=7.5,
    va="top",
    ha="right",
    fontfamily="monospace",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#d0cec8", alpha=0.9),
)
ax = axes[1]
yr_avgs, yr_stds = [], []
for yr in years:
    mask = weekly_r.index.year == yr
    sub = weekly_r[mask].values.flatten()
    sub = sub[~np.isnan(sub)]
    yr_avgs.append(np.mean(sub) * 100)
    yr_stds.append(np.std(sub) * 100)
cols_yr = [ACCENT1 if v > 0 else ACCENT4 for v in yr_avgs]
ax.bar(years, yr_avgs, color=cols_yr, alpha=0.8, width=0.5, zorder=2)
ax.errorbar(years, yr_avgs, yerr=yr_stds, fmt="none", color="#444441", capsize=4, elinewidth=1.5, zorder=3)
ax.axhline(0, color="#888780", lw=1)
for yr, v in zip(years, yr_avgs):
    ax.text(
        yr,
        v + (0.15 if v > 0 else -0.25),
        f"{v:+.2f}%",
        ha="center",
        fontsize=8,
        fontweight="bold",
        color="#2c2c2a",
    )
ax.set_xlabel("Năm")
ax.set_ylabel("Weekly Return trung bình (%)")
ax.set_title("Weekly Return trung bình theo năm\n(± 1 Std — thanh lỗi)")
ax.set_xticks(years)
ax = axes[2]
port_weekly = weekly_r.mean(axis=1)
roll_vol = port_weekly.rolling(12).std() * np.sqrt(52) * 100
ax.fill_between(roll_vol.index, roll_vol.values, color=ACCENT2, alpha=0.25)
ax.plot(roll_vol.index, roll_vol.values, color=ACCENT2, lw=1.5)
ax.axhline(roll_vol.mean(), color=ACCENT3, lw=1.5, ls="--", label=f"Avg = {roll_vol.mean():.1f}%")
covid_idx = roll_vol["2020-03":"2020-05"].idxmax()
ax.annotate(
    "COVID\nselloff",
    xy=(covid_idx, roll_vol[covid_idx]),
    xytext=(30, 8),
    textcoords="offset points",
    fontsize=7.5,
    color=ACCENT4,
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=ACCENT4, lw=1.2),
)
credit_idx = roll_vol["2022-04":"2022-07"].idxmax()
ax.annotate(
    "Credit\ncrunch",
    xy=(credit_idx, roll_vol[credit_idx]),
    xytext=(8, 15),
    textcoords="offset points",
    fontsize=7.5,
    color=ACCENT4,
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=ACCENT4, lw=1.2),
)
ax.set_ylabel("Realized Vol (%, annualized, 12W rolling)")
ax.set_title("Volatility thị trường theo thời gian\n(Equal-weight portfolio, rolling 12 tuần)")
ax.legend(fontsize=8)
plt.tight_layout()
fig4.savefig(FIG_DIR / "fig4_return_dist.png", dpi=180, bbox_inches="tight")
plt.close()
print("Fig4 saved")

# FIGURE 5
fig5, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdYlGn", vmin=-0.1, vmax=0.7, aspect="auto")
ax.set_xticks(range(len(stock_cols)))
ax.set_xticklabels(stock_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(stock_cols)))
ax.set_yticklabels(stock_cols, fontsize=8)
ax.set_title(
    "Ma trận Correlation giữa 26 mã (Weekly Returns, 2019–2024)\n"
    "Xanh đậm = tương quan cao (cùng biến động), Đỏ = ngược chiều"
)
ax.grid(False)
for i in range(len(stock_cols)):
    for j in range(len(stock_cols)):
        v = corr.values[i, j]
        if i != j:
            tc = "white" if v > 0.55 or v < -0.05 else "#2c2c2a"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5.5, color=tc)
cb = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.01)
cb.set_label("Pearson Correlation", fontsize=9)
cb.ax.tick_params(labelsize=8)
plt.tight_layout()
fig5.savefig(FIG_DIR / "fig5_corr_heatmap.png", dpi=180, bbox_inches="tight")
plt.close()
print("Fig5 saved")

# FIGURE 6
fig6, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
miss = df[stock_cols].map(lambda x: pd.isna(clean_price(x))).sum().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
colors_miss = [
    "#E24B4A" if v > 200 else "#BA7517" if v > 50 else "#378ADD" for v in miss_nonzero.values
]
ax1.barh(range(len(miss_nonzero)), miss_nonzero.values, color=colors_miss, height=0.6, alpha=0.85)
ax1.set_yticks(range(len(miss_nonzero)))
ax1.set_yticklabels(miss_nonzero.index, fontsize=9)
ax1.set_xlabel("Số ngày thiếu dữ liệu")
ax1.set_title("Missing Data — Số ngày thiếu theo mã\n(Tổng 1,382 ngày giao dịch)")
for i, v in enumerate(miss_nonzero.values):
    pct = v / len(df) * 100
    ax1.text(v + 3, i, f"{v} ngày ({pct:.1f}%)", va="center", fontsize=8.5, color="#444441")
ax1.set_xlim(0, 720)
ax1.text(
    0.02,
    0.02,
    f"{(miss==0).sum()}/26 mã: dữ liệu đầy đủ",
    transform=ax1.transAxes,
    fontsize=8.5,
    color=ACCENT1,
    fontweight="bold",
    va="bottom",
)
ax2.axis("off")
table_data = [
    ["Chỉ tiêu", "Giá trị", "Ghi chú"],
    ["Mã cổ phiếu", "26", "HOSE — nhóm BĐS & xây dựng"],
    ["Ngày giao dịch", "1,382", "28/06/2019 → 31/12/2024"],
    ["Tuần dữ liệu", "≈ 277", "Weekly aggregation (resample W)"],
    ["Features / mã", "4", "Return, Volatility, Momentum, Price_norm"],
    ["Tensor X", "[286, 26, 4]", "Time × Stocks × Features"],
    ["Tensor Y", "[286, 26, 2]", "Time × Stocks × [min, max return]"],
    ["Window size", "12 tuần", "≈ 3 tháng lookback"],
    ["Walk-forward step", "15 tuần", "≈ 1 quý test window"],
    ["Sequences", "274", "Sau slide window"],
    ["Min train size", "150", "≈ 3 năm lịch sử"],
    ["Test windows", "9", "Expanding window"],
    ["Seeds per window", "3", "42, 52, 62"],
    ["Corr threshold (graph)", "0.7", "Dynamic edge construction"],
    ["Static edges", "214", "Sector-based graph"],
]
tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0], loc="center", cellLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.45)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#e0ddd8")
    if r == 0:
        cell.set_facecolor("#1D9E75")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#f5f5f0")
    else:
        cell.set_facecolor("white")
ax2.set_title("Tóm tắt cấu trúc Dataset & Pipeline\nGAT-LSTM Benchmark", fontsize=11, fontweight="bold", pad=12)
plt.tight_layout()
fig6.savefig(FIG_DIR / "fig6_missing_pipeline.png", dpi=180, bbox_inches="tight")
plt.close()
print("Fig6 saved")
print("\nAll 6 figures saved successfully!")
print("FIG_DIR:", FIG_DIR)


Fig1 saved -> G:\My Drive\MsC_DataScience_UoY_2025_2026\Kevin_Ha\Stock Market\gat_lstm_stock_prediction\dataset\figs\fig1_price_norm.png
Fig2 saved
Fig3 saved
Fig4 saved
Fig5 saved
Fig6 saved

All 6 figures saved successfully!
FIG_DIR: G:\My Drive\MsC_DataScience_UoY_2025_2026\Kevin_Ha\Stock Market\gat_lstm_stock_prediction\dataset\figs


In [7]:
df['Date'] = pd.to_datetime(df['History Price / Date'], dayfirst=False)
df = df.sort_values('Date').reset_index(drop=True)
stock_cols = [c for c in df.columns if c not in ['History Price / Date','Date']]

def clean_price(x):
    if isinstance(x, str):
        x = x.replace(',','').strip()
        if x in ['', '-', 'nan']: return np.nan
        return float(x)
    return x

prices = df[stock_cols].map(clean_price).ffill().bfill()
prices.index = df['Date'].values

# Sector mapping (from GraphConstructor knowledge)
sectors = {
    'Real Estate': ['NVL','PDR','DXG','KDH','NLG','VHM','HDG','HDC','KBC','TDC','TIP','NBB','IJC','CCL','NTL','LHG','SZL','D2D','ITC','TIX','CRE','CDC','VPI','TCH','HPX'],
    'Industrial': ['DIG'],
}
# Actually all 26 are real estate / construction sector on HOSE
# Let's check by IPO / listing date proxy from first non-null
first_date = {}
for s in stock_cols:
    valid = prices[s].dropna()
    if len(valid) > 0:
        first_date[s] = pd.Timestamp(prices.index[prices[s].notna()][0]).date()

print("=== First valid date per stock ===")
for s, d in sorted(first_date.items(), key=lambda x: x[1]):
    print(f"  {s}: {d}")

# Annual returns
print("\n=== Annual return % ===")
for yr in [2020,2021,2022,2023,2024]:
    mask = pd.DatetimeIndex(prices.index).year == yr
    sub = prices[mask]
    if len(sub) < 5: continue
    ret = (sub.iloc[-1] / sub.iloc[0] - 1) * 100
    best = ret.idxmax(); worst = ret.idxmin()
    print(f"{yr}: avg={ret.mean():.1f}% | best={best}({ret[best]:.0f}%) | worst={worst}({ret[worst]:.0f}%)")

# Daily returns & volatility
daily_ret = prices.pct_change().dropna()
vol_ann = daily_ret.std() * np.sqrt(252) * 100

print("\n=== Volatility (annualized %) top/bottom 5 ===")
print("Most volatile:", vol_ann.sort_values(ascending=False).head(5).round(1).to_dict())
print("Least volatile:", vol_ann.sort_values().head(5).round(1).to_dict())

# Overall market return
idx_prices = prices.mean(axis=1)
total_ret = (idx_prices.iloc[-1] / idx_prices.iloc[0] - 1) * 100
print(f"\nEqual-weight portfolio total return 2019→2024: {total_ret:.1f}%")

# Distribution of weekly returns
weekly = prices.resample('W').last().pct_change().dropna()
all_weekly = weekly.values.flatten()
all_weekly = all_weekly[~np.isnan(all_weekly)]
print(f"\nWeekly return distribution:")
print(f"  Mean: {all_weekly.mean()*100:.2f}%")
print(f"  Std:  {all_weekly.std()*100:.2f}%")
print(f"  Min:  {all_weekly.min()*100:.1f}%")
print(f"  Max:  {all_weekly.max()*100:.1f}%")
print(f"  Skew: {pd.Series(all_weekly).skew():.2f}")

# Missing count
miss = df[stock_cols].map(lambda x: pd.isna(clean_price(x))).sum()
print("\n=== Missing trading days ===")
print(miss.sort_values(ascending=False).to_string())

# Sequences stats
print(f"\n=== Model pipeline summary ===")
print(f"Total trading days: {len(df)}")
print(f"Weekly periods approx: {len(df)//5}")
print(f"Window size=12 → sequences ~ {len(df)//5 - 12}")

=== First valid date per stock ===
  CCL: 2019-06-28
  CDC: 2019-06-28
  CRE: 2019-06-28
  D2D: 2019-06-28
  DIG: 2019-06-28
  DXG: 2019-06-28
  HDC: 2019-06-28
  HDG: 2019-06-28
  HPX: 2019-06-28
  IJC: 2019-06-28
  ITC: 2019-06-28
  KBC: 2019-06-28
  KDH: 2019-06-28
  LHG: 2019-06-28
  NBB: 2019-06-28
  NLG: 2019-06-28
  NTL: 2019-06-28
  NVL: 2019-06-28
  PDR: 2019-06-28
  SZL: 2019-06-28
  TCH: 2019-06-28
  TDC: 2019-06-28
  TIP: 2019-06-28
  TIX: 2019-06-28
  VHM: 2019-06-28
  VPI: 2019-06-28

=== Annual return % ===
2020: avg=47.6% | best=TIP(140%) | worst=TCH(-37%)
2021: avg=89.4% | best=DIG(321%) | worst=CDC(-6%)
2022: avg=-53.5% | best=TIX(9%) | worst=HPX(-87%)
2023: avg=32.1% | best=PDR(91%) | worst=CRE(-13%)
2024: avg=6.3% | best=D2D(56%) | worst=NVL(-39%)

=== Volatility (annualized %) top/bottom 5 ===
Most volatile: {'DIG': 52.4, 'DXG': 50.7, 'NBB': 48.2, 'CCL': 48.0, 'HDC': 47.4}
Least volatile: {'VPI': 20.2, 'VHM': 31.5, 'SZL': 31.8, 'KDH': 32.5, 'CDC': 34.7}

Equal-weig

In [11]:
df['Date'] = pd.to_datetime(df['History Price / Date'])
df = df.sort_values('Date').reset_index(drop=True)
stock_cols = [c for c in df.columns if c not in ['History Price / Date','Date']]

def clean_price(x):
    if isinstance(x, str):
        x = x.replace(',','').strip()
        if x in ['','-','nan']: return np.nan
        return float(x)
    return x

prices = df[stock_cols].map(clean_price).ffill().bfill()
prices.index = pd.DatetimeIndex(df['Date'])

# Normalize each stock to 100 at start
norm = prices.div(prices.iloc[0]) * 100

# Per-year returns for each stock
yr_ret = {}
for yr in [2020,2021,2022,2023,2024]:
    mask = prices.index.year == yr
    sub = prices[mask]
    if len(sub) < 5:
        yr_ret[yr] = {s: 0 for s in stock_cols}
    else:
        yr_ret[yr] = ((sub.iloc[-1] / sub.iloc[0] - 1)*100).round(1).to_dict()

# Weekly correlation matrix (for graph insight)
weekly = prices.resample('W').last().pct_change().dropna()
corr = weekly.corr()

# Print correlation stats
print("=== Correlation stats ===")
corr_vals = corr.values[np.triu_indices_from(corr.values, k=1)]
print(f"Mean pairwise corr: {corr_vals.mean():.3f}")
print(f"Std: {corr_vals.std():.3f}")
print(f"Min: {corr_vals.min():.3f}")
print(f"Max: {corr_vals.max():.3f}")

# Top correlated pairs
pairs = []
for i,s1 in enumerate(stock_cols):
    for j,s2 in enumerate(stock_cols):
        if j<=i: continue
        pairs.append((s1,s2,corr.loc[s1,s2]))
pairs.sort(key=lambda x:-x[2])
print("\nTop 5 most correlated pairs:")
for a,b,c in pairs[:5]: print(f"  {a}-{b}: {c:.3f}")
print("\nTop 5 least correlated:")
for a,b,c in pairs[-5:]: print(f"  {a}-{b}: {c:.3f}")

# Normalized price data for charting (monthly sampled)
monthly = norm.resample('ME').last()
print(f"\n=== Normalized price (monthly, base=100) ===")
print(monthly.round(1).to_string())

# Volume proxy: price × daily change (we don't have volume, so use volatility clusters)
daily_ret = prices.pct_change().dropna()

# For each stock: 4 features computed weekly
weekly_prices = prices.resample('W').last()
weekly_ret = weekly_prices.pct_change()
weekly_vol = daily_ret.resample('W').std() * np.sqrt(5)
weekly_mom = weekly_prices.pct_change(4)  # 4-week momentum

print("\n=== 4 features sample (last week) ===")
last = {
    'weekly_return': weekly_ret.iloc[-1].round(4).to_dict(),
    'weekly_volatility': weekly_vol.iloc[-1].round(4).to_dict(),
    '4wk_momentum': weekly_mom.iloc[-1].round(4).to_dict(),
}
for feat, vals in last.items():
    print(f"\n{feat}:")
    for s,v in list(vals.items())[:6]: print(f"  {s}: {v}")

# Min/max return target stats
df_target = pd.read_csv("dataset\min_max_return.csv", header=1)
print("\n=== Target file (min_max_return) shape:", df_target.shape)
print(df_target.head(3))
print("\nTarget value ranges:")
numeric_cols = df_target.select_dtypes(include='number')
print(f"Min return min: {numeric_cols.min().min():.4f}")
print(f"Max return max: {numeric_cols.max().max():.4f}")
print(f"Overall mean: {numeric_cols.mean().mean():.4f}")

=== Correlation stats ===
Mean pairwise corr: 0.319
Std: 0.171
Min: -0.045
Max: 0.688

Top 5 most correlated pairs:
  DIG-DXG: 0.688
  KDH-NLG: 0.657
  DIG-NLG: 0.635
  DXG-KDH: 0.630
  IJC-TDC: 0.626

Top 5 least correlated:
  NBB-TIX: -0.025
  CCL-TIX: -0.029
  PDR-TIX: -0.032
  CDC-HPX: -0.036
  TIX-VPI: -0.045

=== Normalized price (monthly, base=100) ===
              CCL    CDC    CRE    D2D    DIG    DXG     HDC    HDG    HPX    IJC    ITC    KBC    KDH    LHG    NBB    NLG    NTL    NVL    PDR    SZL    TCH    TDC    TIP    TIX    VHM    VPI
Date                                                                                                                                                                                             
2019-06-30  100.0  100.0  100.0  100.0  100.0  100.0   100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
2019-07-31  109.1  106.4   97.3  114.5   97.4   93.0   1

In [6]:
from pathlib import Path
import sys

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

try:
    import reportlab
except ImportError:
    raise ImportError(
        "Missing reportlab. Run the first cell: %pip install reportlab matplotlib"
    ) from None

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm, mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    Table,
    TableStyle,
    HRFlowable,
    PageBreak,
    KeepTogether,
)

W, H = A4

C_DARK = colors.HexColor("#1a1a2e")
C_TEAL = colors.HexColor("#1D9E75")
C_BLUE = colors.HexColor("#378ADD")
C_AMBER = colors.HexColor("#BA7517")
C_CORAL = colors.HexColor("#D85A30")
C_GRAY = colors.HexColor("#888780")
C_LIGHT = colors.HexColor("#f5f5f0")
C_BORDER = colors.HexColor("#e0ddd8")
C_TEXT = colors.HexColor("#2c2c2a")
C_MUTED = colors.HexColor("#5f5e5a")
C_WHITE = colors.white
C_RED = colors.HexColor("#E24B4A")


def project_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "dataset" / "stock_market_19_24.csv").exists():
            return cand
    raise FileNotFoundError("Không tìm thấy dataset/stock_market_19_24.csv")


ROOT = project_root()
FIG_DIR = ROOT / "dataset" / "figs"
OUT = ROOT / "dataset" / "HOSE_Dataset_Analysis.pdf"


def S(name, **kw):
    return ParagraphStyle(name, **kw)


sH1 = S(
    "sH1",
    fontSize=15,
    leading=20,
    textColor=C_DARK,
    spaceBefore=16,
    spaceAfter=5,
    fontName="Helvetica-Bold",
)
sH2 = S(
    "sH2",
    fontSize=10,
    leading=14,
    textColor=C_TEAL,
    spaceBefore=10,
    spaceAfter=4,
    fontName="Helvetica-Bold",
)
sBody = S(
    "sBody",
    fontSize=9.5,
    leading=15,
    textColor=C_TEXT,
    spaceAfter=5,
    fontName="Helvetica",
    alignment=TA_JUSTIFY,
)
sBullet = S(
    "sBullet",
    fontSize=9.5,
    leading=15,
    textColor=C_TEXT,
    spaceAfter=3,
    fontName="Helvetica",
    leftIndent=14,
)
sCaption = S(
    "sCaption",
    fontSize=7.5,
    leading=11,
    textColor=C_MUTED,
    spaceAfter=8,
    fontName="Helvetica",
    alignment=TA_CENTER,
)
sInsight = S(
    "sInsight",
    fontSize=9,
    leading=13,
    textColor=colors.HexColor("#0C447C"),
    spaceAfter=4,
    fontName="Helvetica",
    leftIndent=10,
    rightIndent=10,
    backColor=colors.HexColor("#E6F1FB"),
    borderPad=6,
)
sHL = S(
    "sHL",
    fontSize=9,
    leading=13,
    textColor=colors.HexColor("#085041"),
    spaceAfter=4,
    fontName="Helvetica-Bold",
    leftIndent=10,
    rightIndent=10,
    backColor=colors.HexColor("#E1F5EE"),
    borderPad=6,
)


def alt_style(tbl_obj, n_data_rows, hdr_bg=C_DARK, hdr_fg=C_WHITE):
    cmds = [
        ("BACKGROUND", (0, 0), (-1, 0), hdr_bg),
        ("TEXTCOLOR", (0, 0), (-1, 0), hdr_fg),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
        ("ALIGN", (0, 0), (-1, -1), "LEFT"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("GRID", (0, 0), (-1, -1), 0.4, C_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
    ]
    for r in range(1, n_data_rows + 1):
        bg = C_LIGHT if r % 2 == 0 else C_WHITE
        cmds.append(("BACKGROUND", (0, r), (-1, r), bg))
    tbl_obj.setStyle(TableStyle(cmds))


def build_header_footer(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(C_DARK)
    canvas.rect(0, H - 28 * mm, W, 28 * mm, fill=1, stroke=0)
    canvas.setFillColor(C_TEAL)
    canvas.rect(0, H - 30 * mm, W, 2 * mm, fill=1, stroke=0)
    canvas.setFillColor(C_WHITE)
    canvas.setFont("Helvetica-Bold", 10)
    canvas.drawString(2 * cm, H - 18 * mm, "HOSE Dataset Analysis — Quantitative Report")
    canvas.setFont("Helvetica", 8)
    canvas.drawRightString(W - 2 * cm, H - 18 * mm, "GAT-LSTM Research | 2024")
    canvas.setFillColor(C_LIGHT)
    canvas.rect(0, 0, W, 18 * mm, fill=1, stroke=0)
    canvas.setFillColor(C_BORDER)
    canvas.rect(0, 18 * mm, W, 0.3 * mm, fill=1, stroke=0)
    canvas.setFillColor(C_MUTED)
    canvas.setFont("Helvetica", 7.5)
    canvas.drawString(2 * cm, 7 * mm, "Confidential — For Research Purposes Only")
    canvas.drawRightString(W - 2 * cm, 7 * mm, f"Page {doc.page}")
    canvas.restoreState()


def build_cover(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(C_DARK)
    canvas.rect(0, 0, W, H, fill=1, stroke=0)
    canvas.setFillColor(C_TEAL)
    canvas.rect(0, H * 0.44, W, 3 * mm, fill=1, stroke=0)
    canvas.rect(2 * cm, H * 0.44, 1.5 * mm, H * 0.48, fill=1, stroke=0)
    canvas.restoreState()


doc = SimpleDocTemplate(
    str(OUT),
    pagesize=A4,
    leftMargin=2 * cm,
    rightMargin=2 * cm,
    topMargin=3.5 * cm,
    bottomMargin=2.5 * cm,
)
story = []

story.append(Spacer(1, 4.5 * cm))
story.append(
    Paragraph(
        '<font color="#FFFFFF" size="30"><b>HOSE Market Dataset</b></font>',
        S("ct1", fontSize=30, leading=36, fontName="Helvetica-Bold", textColor=C_WHITE),
    )
)
story.append(
    Paragraph(
        '<font color="#1D9E75" size="30"><b>Quantitative Analysis</b></font>',
        S("ct2", fontSize=30, leading=38, fontName="Helvetica-Bold", textColor=C_TEAL, spaceAfter=12),
    )
)
story.append(
    Paragraph(
        '<font color="#9FE1CB">26 HOSE Real Estate Stocks &nbsp;·&nbsp; 2019 – 2024 &nbsp;·&nbsp; GAT-LSTM Research</font>',
        S("cs", fontSize=12, fontName="Helvetica", textColor=colors.HexColor("#9FE1CB"), spaceAfter=14),
    )
)
story.append(
    Paragraph(
        '<font color="#888780">Quantitative analysis report of the dataset structure for stock return forecasting research based on Graph Attention Networks combined with LSTM. Includes descriptive statistics, volatility analysis, inter-stock correlation, regime analysis, and data quality assessment.</font>',
        S(
            "cd",
            fontSize=9.5,
            leading=15,
            fontName="Helvetica",
            textColor=C_GRAY,
            alignment=TA_JUSTIFY,
            rightIndent=2 * cm,
            spaceAfter=24,
        ),
    )
)

cov_data = [["1,382", "26", "274", "4"], ["Trading days", "Stock codes", "Sequences", "Features"]]
tc = Table(cov_data, colWidths=[3.8 * cm] * 4)
tc.setStyle(
    TableStyle(
        [
            ("FONTNAME", (0, 0), (3, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (3, 0), 22),
            ("TEXTCOLOR", (0, 0), (3, 0), C_TEAL),
            ("FONTNAME", (0, 1), (3, 1), "Helvetica"),
            ("FONTSIZE", (0, 1), (3, 1), 8),
            ("TEXTCOLOR", (0, 1), (3, 1), C_GRAY),
            ("ALIGN", (0, 0), (3, 1), "CENTER"),
            ("TOPPADDING", (0, 0), (3, 1), 6),
            ("BOTTOMPADDING", (0, 0), (3, 1), 6),
            ("LINEBELOW", (0, 0), (3, 0), 0.5, colors.HexColor("#444441")),
        ]
    )
)
story.append(tc)
story.append(Spacer(1, 4 * cm))
story.append(
    Paragraph(
        '<font color="#444441">Ho Chi Minh City Stock Exchange (HOSE) &nbsp;·&nbsp; Real Estate & Construction Sector</font>',
        S("cf", fontSize=8, fontName="Helvetica", textColor=C_GRAY),
    )
)
story.append(PageBreak())

story.append(Paragraph("1. Dataset Overview", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(
    Paragraph(
        "The dataset is constructed from the daily closing prices of 26 real estate and construction stocks listed on HOSE, covering the period from June 28, 2019, to December 31, 2024 (1,382 trading sessions). Data is aggregated weekly, with 4 features calculated per week (weekly return, volatility, 4-week momentum, and normalized price), creating an input tensor of shape [286, 26, 4]. A walk-forward expanding window strategy with a step of 15 weeks generates 274 sequences for 9 test windows.",
        sBody,
    )
)

sum_data = [
    ["Parameter", "Value", "Parameter", "Value"],
    ["No. of Stocks", "26", "Exchange", "HOSE"],
    ["Sector", "RE & Construction", "Timeframe", "06/2019–12/2024"],
    ["Trading Days", "1,382", "Total Weeks", "≈277"],
    ["Features/Stock", "4/week", "X Tensor Shape", "[286, 26, 4]"],
    ["Y Tensor Shape", "[286, 26, 2]", "Walk-forward step", "15 weeks (1 qtr)"],
    ["Window size", "12 weeks", "Sequences", "274"],
    ["Min train size", "150 seq", "Test windows", "9"],
    ["Corr threshold", "0.7 (dyn. edge)", "Static edges", "214 (sector)"],
]
t1 = Table(sum_data, colWidths=[4.5 * cm] * 4)
alt_style(t1, len(sum_data) - 1)
t1.setStyle(
    TableStyle(
        [
            ("BACKGROUND", (0, 0), (3, 0), C_DARK),
            ("TEXTCOLOR", (0, 0), (3, 0), C_WHITE),
            ("FONTNAME", (0, 0), (3, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (3, -1), 8.5),
            ("ALIGN", (0, 0), (3, -1), "LEFT"),
            ("VALIGN", (0, 0), (3, -1), "MIDDLE"),
            ("GRID", (0, 0), (3, -1), 0.4, C_BORDER),
            ("TOPPADDING", (0, 0), (3, -1), 5),
            ("BOTTOMPADDING", (0, 0), (3, -1), 5),
            ("LEFTPADDING", (0, 0), (3, -1), 8),
            *[
                ("BACKGROUND", (0, r), (3, r), C_LIGHT if r % 2 == 0 else C_WHITE)
                for r in range(1, len(sum_data))
            ],
        ]
    )
)
story.append(Spacer(1, 6))
story.append(t1)
story.append(PageBreak())

story.append(Paragraph("2. Price Trends — Normalized to Day One", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(Image(str(FIG_DIR / "fig1_price_norm.png"), width=16 * cm, height=6.5 * cm))
story.append(
    Paragraph(
        "Figure 1: Normalized prices of 26 HOSE stocks (base=100 as of 2019-06-28). Gray lines = baseline stocks. Colors highlight specific growth/decline patterns. Shaded regions: COVID selloff, 2021 Bubble, 2022 Credit crunch.",
        sCaption,
    )
)
story.append(Paragraph("Analysis:", sH2))
story.append(
    Paragraph(
        "Extreme divergence between stocks in the same sector after 5.5 years is the most prominent feature. HDC peaked at ~10x while VHM — the largest market cap stock in the group — fell to 0.69x. The market experienced three distinct regimes:",
        sBody,
    )
)
for b in [
    '<b>Q1/2020 — COVID Selloff:</b> Synchronized sell-off for 6 weeks. HPX and CCL suffered the most due to poor liquidity. Correlations spiked toward 1.0 — the "everything falls together" effect.',
    "<b>2021 — RE Bubble:</b> All 26 stocks surged, DIG +321%, HDC +916%. This is a phase where graph models hold an advantage due to rapid signal propagation across the system. Mean correlation reached its peak.",
    "<b>2022 — Credit Crunch:</b> The Tan Hoang Minh event and corporate bond tightening caused a sector-wide collapse. NVL -85%, HPX -87%. Only TIX escaped (+9%) due to suspension. This extreme tail risk is the ultimate challenge for any model.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(Spacer(1, 4))
story.append(
    Paragraph(
        "Model Insight: Massive divergence proves a static sector graph is insufficient. Dynamic correlation edges (updated per window) are necessary to reflect stocks decoupling during crises.",
        sInsight,
    )
)
story.append(PageBreak())

story.append(Paragraph("3. Annual Return Heatmap — 26 Stocks × 5 Years", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(Image(str(FIG_DIR / "fig2_annual_heatmap.png"), width=14 * cm, height=9 * cm))
story.append(
    Paragraph(
        "Figure 2: Annual Return Heatmap (%). Dark green = high profit, dark red = heavy loss. The 'avg' column on the right is the 5-year average per stock.",
        sCaption,
    )
)

obs_data = [
    ["Year", "Avg", "Best Stock", "Worst Stock", "Commentary"],
    ["2020", "+47.6%", "TIP (+140%)", "TCH (-37%)", "Post-COVID recovery. Clear divergence."],
    ["2021", "+89.4%", "DIG (+321%)", "CDC (-6%)", "Full-blown bubble. 25/26 stocks positive."],
    ["2022", "-53.5%", "TIX (+9%)", "HPX (-87%)", "Crisis. Only 1 stock escaped loss."],
    ["2023", "+32.1%", "PDR (+91%)", "CRE (-13%)", "Uneven recovery."],
    ["2024", "+6.3%", "D2D (+56%)", "NVL (-39%)", "Sideways, highly personalized performance."],
]
t3 = Table(obs_data, colWidths=[1.5 * cm, 1.8 * cm, 3 * cm, 3 * cm, 6.2 * cm])
t3.setStyle(
    TableStyle(
        [
            ("BACKGROUND", (0, 0), (4, 0), C_TEAL),
            ("TEXTCOLOR", (0, 0), (4, 0), C_WHITE),
            ("FONTNAME", (0, 0), (4, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (4, -1), 8),
            ("ALIGN", (0, 0), (4, -1), "LEFT"),
            ("VALIGN", (0, 0), (4, -1), "MIDDLE"),
            *[("BACKGROUND", (0, r), (4, r), C_LIGHT if r % 2 == 0 else C_WHITE) for r in range(1, 6)],
            ("GRID", (0, 0), (4, -1), 0.4, C_BORDER),
            ("TOPPADDING", (0, 0), (4, -1), 5),
            ("BOTTOMPADDING", (0, 0), (4, -1), 5),
            ("LEFTPADDING", (0, 0), (4, -1), 6),
            ("TEXTCOLOR", (1, 1), (1, 1), C_TEAL),
            ("TEXTCOLOR", (1, 2), (1, 2), C_TEAL),
            ("TEXTCOLOR", (1, 3), (1, 3), C_RED),
            ("TEXTCOLOR", (1, 4), (1, 4), C_TEAL),
            ("TEXTCOLOR", (1, 5), (1, 5), C_TEAL),
            ("FONTNAME", (1, 1), (1, -1), "Helvetica-Bold"),
        ]
    )
)
story.append(t3)
story.append(Spacer(1, 4))
story.append(
    Paragraph(
        "Key Observation: Cross-sectional variance (between stocks) is greater than time-series variance (same stock over years) — evidence that stock-specific factors outweigh general market trends. This is why models require stock-specific embeddings and should avoid simple full-batch loss averaging.",
        sInsight,
    )
)
story.append(PageBreak())

story.append(Paragraph("4. Volatility & Correlation Analysis", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(Image(str(FIG_DIR / "fig3_vol_corr.png"), width=16 * cm, height=6 * cm))
story.append(
    Paragraph(
        "Figure 3. Left: Annualized volatility ranking (26 stocks). Right: Distribution of Pearson correlations between 325 stock pairs (weekly returns 2019–2024).",
        sCaption,
    )
)
story.append(Paragraph("Volatility:", sH2))
for b in [
    "<b>High-risk group (>45%): DIG, DXG, NBB, CCL, HDC</b> — wide fluctuation range, difficult to predict but carries the most signal for graph models. Requires per-stock loss weighting to avoid gradient domination.",
    "<b>Stable group (<35%): VPI, VHM, SZL, KDH</b> — easier to predict but lower alpha. Models tend to overfit this group, though most returns come from the volatile group.",
    "<b>20%–52% Gap</b> (2.6x difference) necessitates normalized targets or weighted loss to prevent the model from ignoring stable stocks in favor of volatile ones.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(Paragraph("Correlation:", sH2))
for b in [
    "<b>Mean = 0.319, Std = 0.171:</b> Moderate average correlation — ideal for GAT. Not too high (prevents message passing from becoming simple averaging) and not too low (graph still retains signals).",
    "<b>Top pairs (DIG-DXG: 0.691, KDH-NLG: 0.655):</b> Stocks in similar segments (Industrial vs. Residential) or regions. Confirms geographic/sector graph hypothesis.",
    "<b>TIX shows negative correlations with many stocks</b> (-0.045 vs VPI): This is noise from 615 days of missing data, not a real signal. Needs separate handling.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(Spacer(1, 4))
story.append(
    Paragraph(
        "Conclusion: A dynamic graph (updating correlations per window) is superior to a static graph because correlations shift significantly between regimes — especially during the 2022 crisis when pairs decoupled.",
        sHL,
    )
)
story.append(PageBreak())

story.append(Paragraph("5. Full 26×26 Correlation Matrix", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(Image(str(FIG_DIR / "fig5_corr_heatmap.png"), width=15.5 * cm, height=12.5 * cm))
story.append(
    Paragraph(
        "Figure 4: 26×26 Correlation Matrix (weekly returns, 2019–2024). Dark blue = high correlation, red = negative correlation.",
        sCaption,
    )
)
story.append(Paragraph("Cluster Analysis:", sH2))
for b in [
    "<b>Industrial RE Cluster:</b> DIG, DXG, NLG (0.63–0.69). Responds in tandem to FDI and industrial land prices.",
    "<b>Residential RE Cluster:</b> KDH, NLG, KBC (0.55–0.65). Driven by shared residential credit cycles.",
    "<b>Outliers (pale rows/cols): TIX, HPX, CDC</b> — missing data skewing correlation. Edges from these stocks should be downweighted in the graph.",
    "<b>VHM and NVL:</b> Despite being the largest market caps, their correlation with the rest is only 0.2–0.4. Driven by specific macro factors (restructuring, scandals) not shared with SME RE stocks.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(PageBreak())

story.append(Paragraph("6. Return Distribution & Regime Analysis", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(Image(str(FIG_DIR / "fig4_return_dist.png"), width=16 * cm, height=5.5 * cm))
story.append(
    Paragraph(
        "Figure 5. Left: Weekly return histogram (N≈7,100 obs). Middle: Average weekly return by year ±1 Std. Right: Rolling 12-week realized volatility.",
        sCaption,
    )
)

s_data = [
    ["Metric", "Value", "Significance"],
    ["Mean", "+0.33%/week", "≈17%/year — typical for emerging markets"],
    ["Std", "5.95%/week", "≈43% annualized — very high risk"],
    ["Min", "-30.2%/week", "Tail risk: COVID selloff Q1/2020"],
    ["Max", "+38.8%/week", "Extreme upside: penny stocks boom 2021"],
    ["Skewness", "≈ 0.02", "Nearly symmetric — not heavily left-skewed"],
    ["Kurtosis", "~6–8 (est.)", "Fat-tailed: outliers more frequent than normal distribution"],
]
ts = Table(s_data, colWidths=[3.5 * cm, 4 * cm, 9 * cm])
ts.setStyle(
    TableStyle(
        [
            ("BACKGROUND", (0, 0), (2, 0), C_DARK),
            ("TEXTCOLOR", (0, 0), (2, 0), C_WHITE),
            ("FONTNAME", (0, 0), (2, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (2, -1), 8.5),
            ("ALIGN", (0, 0), (2, -1), "LEFT"),
            ("VALIGN", (0, 0), (2, -1), "MIDDLE"),
            *[("BACKGROUND", (0, r), (2, r), C_LIGHT if r % 2 == 0 else C_WHITE) for r in range(1, 7)],
            ("FONTNAME", (0, 1), (0, -1), "Helvetica-Bold"),
            ("TEXTCOLOR", (0, 1), (0, -1), C_MUTED),
            ("TEXTCOLOR", (1, 1), (1, -1), colors.HexColor("#185FA5")),
            ("FONTNAME", (1, 1), (1, -1), "Helvetica-Bold"),
            ("GRID", (0, 0), (2, -1), 0.4, C_BORDER),
            ("TOPPADDING", (0, 0), (2, -1), 5),
            ("BOTTOMPADDING", (0, 0), (2, -1), 5),
            ("LEFTPADDING", (0, 0), (2, -1), 8),
        ]
    )
)
story.append(ts)
story.append(Spacer(1, 6))
story.append(Paragraph("Model Design Implications:", sH2))
for b in [
    "<b>Fat-tailed distribution:</b> MSELoss heavily penalizes outliers (squared error), making the model avoid extreme cases rather than forecasting them. This is why IC loss (rank-based) is preferred over MSE.",
    "<b>Regime-dependent volatility:</b> Rolling vol spiked during COVID and the credit crunch. Walk-forward validation correctly reflects this, but the model needs mechanism to identify regime shifts.",
    "<b>Weekly std = 5.95% >> daily std/sqrt(5):</b> Indicates autocorrelation in intraweek trading. Daily-level features (volatility, momentum) provide info that weekly price alone misses.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(Spacer(1, 4))
story.append(
    Paragraph(
        "Regime Conclusion: The difference between 2021 (+89%, low vol) and 2022 (-54%, high vol) proves a model trained on 2019–2021 will fail in 2022. Walk-forward expanding windows (min_train_size=150) is the only valid validation strategy.",
        sHL,
    )
)
story.append(PageBreak())

story.append(Paragraph("7. Data Quality & Pipeline Structure", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=6))
story.append(Image(str(FIG_DIR / "fig6_missing_pipeline.png"), width=16 * cm, height=5.5 * cm))
story.append(
    Paragraph(
        "Figure 6. Left: Number of missing data days by stock. Right: Summary of dataset structure and pipeline.",
        sCaption,
    )
)
story.append(Paragraph("Data Quality Assessment:", sH2))
for b in [
    "<b>TIX: 615 missing days (44.5%)</b> — Severe. Forward-fill creates artificial 'flat' prices, skewing correlation. Recommendation: remove TIX from graph edges or apply a validity mask.",
    "<b>HPX: 125 missing days (9.0%)</b> — Concentrated in late 2022–early 2023, coinciding with suspension due to consecutive floor drops.",
    "<b>CDC: 86 missing days (6.2%)</b> — Mostly early in the 2019 dataset. Forward-fill is acceptable as trading stabilized later.",
    "<b>20 remaining stocks: 0 missing days</b> — Clean data, ready for direct training.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(Paragraph("Pipeline Improvement Recommendations:", sH2))
for b in [
    "Implement <b>validity masks</b>: a boolean tensor [T, N] to mask loss computation and avoid gradients from forward-filled data.",
    "Calculate <b>excess return relative to VN-Index</b> (alpha) to strip away general market beta and focus on stock-specific signals.",
    "<b>Winsorize returns</b> at the 99th percentile (≈±20%/week) to mitigate the impact of extreme tail risk on the loss function.",
    "Add <b>turnover ratio</b> if volume data is available — a key liquidity proxy used in MDGNN architectures.",
]:
    story.append(Paragraph(f"• {b}", sBullet))
story.append(PageBreak())

story.append(Paragraph("8. Summary & Recommendations", sH1))
story.append(HRFlowable(width="100%", thickness=0.5, color=C_BORDER, spaceAfter=8))
story.append(
    Paragraph(
        "The HOSE 26 Real Estate dataset (2019–2024) is highly complex due to extreme volatility, clear regime shifts, and rich correlation structures. Below are the summarized challenges and opportunities for the GAT-LSTM model:",
        sBody,
    )
)

takeaways = [
    (
        "Challenge",
        "Extreme Regime Shifts",
        C_RED,
        colors.HexColor("#FEF3F2"),
        "Three market phases (bubble/crisis/recovery) with distinct properties. Walk-forward validation is mandatory. Do not rely on aggregate MAE — analyze per-window.",
    ),
    (
        "Challenge",
        "Fat-tailed distribution",
        C_RED,
        colors.HexColor("#FEF3F2"),
        "High kurtosis. MSELoss is unsuitable — use IC loss (Pearson rank-based) to optimize ranking. This explains why GAT-LSTM won 0/26 stocks in the final window.",
    ),
    (
        "Challenge",
        "Large Cross-sectional Divergence",
        C_RED,
        colors.HexColor("#FEF3F2"),
        "Inter-stock variance >> time-series variance. Stock-specific embedding is mandatory. Full-batch loss averaging suppresses gradients from less volatile stocks.",
    ),
    (
        "Opportunity",
        "Learnable Graph Structure",
        C_TEAL,
        colors.HexColor("#ECFDF5"),
        "Mean correlation 0.319 with no skew. Dynamic correlation graphs (updated per window) can detect clusters shifting with regimes — GAT's main edge over LSTM.",
    ),
    (
        "Opportunity",
        "Meaningful Sector Clustering",
        C_TEAL,
        colors.HexColor("#ECFDF5"),
        "Industrial RE (DIG-DXG-NLG) and Residential RE (KDH-NLG-KBC) clusters are distinct. Multi-relational graphs (sector + correlation) per MDGNN can exploit this.",
    ),
    (
        "Risk",
        "Data Quality Issues",
        C_AMBER,
        colors.HexColor("#FFFBEB"),
        "TIX missing 44.5% of sessions. Forward-fill creates fake correlations. Requires validity masking and edge downweighting for TIX/HPX.",
    ),
]

for typ, title, border_c, bg_c, desc in takeaways:
    td = [
        [
            Paragraph(f"<b>{title}</b>", S("ti", fontSize=9, fontName="Helvetica-Bold", textColor=C_TEXT)),
            Paragraph(desc, S("td", fontSize=8.5, fontName="Helvetica", textColor=C_TEXT, leading=13)),
        ]
    ]
    tr = Table(td, colWidths=[3.8 * cm, 12 * cm])
    tr.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (0, 0), bg_c),
                ("BACKGROUND", (1, 0), (1, 0), C_WHITE),
                ("LINEAFTER", (0, 0), (0, 0), 2.5, border_c),
                ("BOX", (0, 0), (1, 0), 0.5, C_BORDER),
                ("VALIGN", (0, 0), (1, 0), "TOP"),
                ("TOPPADDING", (0, 0), (1, 0), 8),
                ("BOTTOMPADDING", (0, 0), (1, 0), 8),
                ("LEFTPADDING", (0, 0), (1, 0), 10),
                ("RIGHTPADDING", (0, 0), (1, 0), 10),
            ]
        )
    )
    story.append(tr)
    story.append(Spacer(1, 4))

story.append(Spacer(1, 8))
story.append(
    Paragraph(
        "Three highest priority improvements: (1) Switch to IC loss to optimize ranking instead of absolute error; (2) Add stock-specific embeddings to differentiate tickers; (3) Apply multi-relational graphs (sector + dynamic correlation) per MDGNN. These directly address the issue where GAT-LSTM won 0/26 stocks in the final window despite ranking #2 in aggregate MAE.",
        sHL,
    )
)

doc.build(story, onFirstPage=build_cover, onLaterPages=build_header_footer)
print(f"PDF saved! {OUT} — {OUT.stat().st_size/1024:.0f} KB")


PDF saved! G:\My Drive\MsC_DataScience_UoY_2025_2026\Kevin_Ha\Stock Market\gat_lstm_stock_prediction\dataset\HOSE_Dataset_Analysis.pdf — 1809 KB
